In [ ]:
# =========================
# 0) Setup (Colab)
# =========================
!pip install -U scikit-learn numpy pandas scipy joblib

import csv
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List, Tuple
import json
import joblib

from google.colab import drive
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, make_scorer,
    classification_report, confusion_matrix
)

# =========================
# 1) Config & Drive
# =========================
drive.mount('/content/drive')
csv_path = "/content/drive/MyDrive/K2/k2pandc.csv"

label_col       = "disposition"
keep_asymmetric = False
min_den         = 1e-12
random_state    = 42
test_size       = 0.20
missing_threshold = 0.70  # drop features with >70% missing

# K2-specific drops
k2_id_ref_cols = [
    "pl_name", "hostname", "disp_refname", "pl_refname", "st_refname", "sy_refname"
]
k2_coord_cols  = ["rastr", "ra", "decstr", "dec"]
k2_date_cols   = ["rowupdate", "pl_pubdate", "releasedate"]
k2_misc_drop   = ["default_flag", "pl_controv_flag"]
cols_to_drop   = k2_id_ref_cols + k2_coord_cols + k2_date_cols + k2_misc_drop

drop_lim_cols = True
header_identifiers = ["pl_orbper", "pl_rade", "st_teff", label_col]

# =========================
# 2) Helpers
# =========================
def detect_header_row(path: str, identifiers: List[str], max_lines: int = 200) -> int:
    with open(path, "r", newline="", encoding="utf-8") as f:
        reader = csv.reader(f)
        for i, row in enumerate(reader):
            if i >= max_lines:
                break
            norm = [str(c).strip().lower() for c in row]
            for token in identifiers:
                if str(token).strip().lower() in norm:
                    return i
    raise ValueError(
        f"Could not detect header within first {max_lines} lines. "
        f"Update 'header_identifiers' with a reliable column name."
    )

def add_uncertainty_features(df_in: pd.DataFrame,
                             keep_asymmetric: bool = False,
                             min_den: float = 1e-12) -> Tuple[pd.DataFrame, list]:
    df_out = df_in.copy()

    err1_cols = [c for c in df_out.columns if c.endswith("_err1")]
    base_to_errs = {}
    for e1 in err1_cols:
        base = e1[:-5]
        e2 = f"{base}_err2"
        if e2 in df_out.columns:
            base_to_errs[base] = (e1, e2)

    new_cols = {}
    for base, (e1, e2) in base_to_errs.items():
        if base not in df_out.columns:
            continue
        val  = pd.to_numeric(df_out[base], errors="coerce")
        err1 = pd.to_numeric(df_out[e1], errors="coerce").abs()
        err2 = pd.to_numeric(df_out[e2], errors="coerce").abs()

        abs_err_mean = (err1 + err2) / 2.0
        denom = np.maximum(np.abs(val), min_den)
        rel_err = abs_err_mean / denom
        new_cols[f"{base}_rel_error"] = rel_err

        if keep_asymmetric:
            new_cols[f"{base}_rel_err_upper"] = err1 / denom
            new_cols[f"{base}_rel_err_lower"] = err2 / denom

    for k, v in new_cols.items():
        df_out[k] = v

    drop_cols = []
    for base, (e1, e2) in base_to_errs.items():
        drop_cols.extend([e1, e2])
    df_out = df_out.drop(columns=drop_cols, errors="ignore")

    return df_out, sorted(new_cols.keys())

# =========================
# 3) Load & base cleaning
# =========================
hdr = detect_header_row(csv_path, header_identifiers)
print(f"Detected header row at line index: {hdr}")

df = pd.read_csv(csv_path, header=hdr)
print("Raw shape:", df.shape)

# Drop empty rows/cols
df.dropna(how="all", inplace=True)
df.dropna(axis=1, how="all", inplace=True)
print(f"After empty drops: {df.shape}")

# Drop K2 non-predictive cols and *_lim
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors="ignore")
if drop_lim_cols:
    lim_cols = [c for c in df.columns if c.endswith("lim")]
    df = df.drop(columns=lim_cols, errors="ignore")
    print(f"Dropped *_lim columns: {len(lim_cols)}")

print(f"After K2 drops: {df.shape}")

# Add uncertainty features
df_u, added_unc_cols = add_uncertainty_features(df, keep_asymmetric=keep_asymmetric, min_den=min_den)
print("Added uncertainty columns:", len(added_unc_cols))

# Remove rows with missing labels
df_u = df_u.dropna(axis=0, subset=[label_col])

# One-hot encode relevant categoricals if present
categorical_cols = [c for c in ["discoverymethod", "disc_facility", "soltype"] if c in df_u.columns]
if categorical_cols:
    df_u = pd.get_dummies(df_u, columns=categorical_cols, dummy_na=True)

# =========================
# 4) Features/labels, numeric coercion, prune & flags (moved BEFORE split)
# =========================
y_raw = df_u[label_col].astype(str).values
X_raw = df_u.drop(columns=[label_col]).copy()

# Numeric coercion
X_raw = X_raw.apply(pd.to_numeric, errors="coerce")

# Drop constant/all-NaN
constant_cols = [c for c in X_raw.columns if X_raw[c].nunique(dropna=True) <= 1]
if constant_cols:
    X_raw = X_raw.drop(columns=constant_cols)
    print("Dropped constant columns:", constant_cols)

# ---- IMPORTANT: prune and add missingness flags BEFORE split ----
drop_cols_miss = [c for c in X_raw.columns if X_raw[c].isna().mean() > missing_threshold]
X_raw = X_raw.drop(columns=drop_cols_miss)
print(f"Dropped {len(drop_cols_miss)} columns with >{int(missing_threshold*100)}% missing")

for col in X_raw.columns:
    if X_raw[col].isna().any():
        X_raw[f"{col}_was_missing"] = X_raw[col].isna().astype(int)
import json

# X_raw must be the exact feature frame you trained the pipeline on
with open("feature_names.json", "w", encoding="utf-8") as f:
    json.dump(X_raw.columns.tolist(), f)

print(len(X_raw.columns), "feature names saved")

# Encode labels
le = LabelEncoder()
y = le.fit_transform(y_raw)
print("Classes:", list(le.classes_))

# Train/Test split AFTER all feature shaping
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=test_size, random_state=random_state, stratify=y
)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

# =========================
# 5) Pipeline (RF only) + CV
# =========================
imputer = SimpleImputer(strategy="median")

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=5,               # tune if needed
    min_samples_split=5,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=random_state,
)

pipe = Pipeline(steps=[
    ("imputer", imputer),
    ("clf", rf),
])

scoring = {
    "accuracy": make_scorer(accuracy_score),
    "f1_macro": make_scorer(f1_score, average="macro"),
    "balanced_acc": "balanced_accuracy",
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)

cv_results = cross_validate(
    pipe,
    X_train, y_train,
    scoring=scoring,
    cv=cv,
    n_jobs=-1,
    return_train_score=False,
)

print("\nRandom Forest (no scalers) cross-validation:")
print(f"Mean Accuracy:     {np.mean(cv_results['test_accuracy']):.4f} ± {np.std(cv_results['test_accuracy']):.4f}")
print(f"Mean Macro-F1:     {np.mean(cv_results['test_f1_macro']):.4f} ± {np.std(cv_results['test_f1_macro']):.4f}")
print(f"Mean Balanced-Acc: {np.mean(cv_results['test_balanced_acc']):.4f} ± {np.std(cv_results['test_balanced_acc']):.4f}")

# =========================
# 6) Fit & evaluate on held-out test
# =========================
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

print("\nHeld-out test set performance (Random Forest only):")
print("Confusion matrix (counts):")
print(confusion_matrix(y_test, y_pred))
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=list(le.classes_)))

# =========================
# 7) Save artifacts (+ feature order)
# =========================
joblib.dump(pipe, "k2_randomforest_pipeline.joblib")
joblib.dump(le,   "k2_randomforest_label_encoder.joblib")
with open("feature_names.json", "w", encoding="utf-8") as f:
    json.dump(X_raw.columns.tolist(), f)

print("\nSaved: k2_randomforest_pipeline.joblib, k2_randomforest_label_encoder.joblib, feature_names.json")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Detected header row at line index: 98
Raw shape: (4004, 94)
After empty drops: (4004, 94)
Dropped *_lim columns: 14
After K2 drops: (4004, 65)
Added uncertainty columns: 0
Dropped constant columns: ['pl_bmassprov', 'st_spectype', 'st_metratio', 'discoverymethod_nan', 'disc_facility_nan', 'soltype_nan']
Dropped 18 columns with >70% missing
92 feature names saved
Classes: ['CANDIDATE', 'CONFIRMED', 'FALSE POSITIVE', 'REFUTED']
Train shape: (3203, 92) Test shape: (801, 92)

Random Forest (no scalers) cross-validation:
Mean Accuracy:     0.9116 ± 0.0139
Mean Macro-F1:     0.7741 ± 0.0416
Mean Balanced-Acc: 0.7752 ± 0.0610

Held-out test set performance (Random Forest only):
Confusion matrix (counts):
[[224   9  40   2]
 [  1 461   1   0]
 [  8   1  49   1]
 [  2   1   0   1]]

Classification report:
                precision    recall  f1-score   support

     CA